# RoseTTAFold3 Structure Prediction

![RoseTTAFold3 Structure Prediction](https://proto-bio.github.io/proto-assets/images/tool/rf3/hero.png)

This notebook demonstrates all-atom structure prediction with RoseTTAFold3 (RF3) from the [Institute for Protein Design](https://www.ipd.uw.edu/). RF3 predicts 3D structures of proteins, DNA, RNA, small-molecule ligands, and their complexes via a diffusion-based generative model with explicit chirality handling. We walk through `run_rf3_prediction` on a small protein-ligand complex, covering input construction, configuration, result inspection, and export.

In [ ]:
from proto_tools.utils.notebook_docs import (
    display_api_reference,
    display_available_tools,
    display_doc_link,
    display_docs_section,
    display_overview,
)

display_doc_link("rf3")
display_overview("rf3")
display_docs_section("rf3", "Background")

## Available tools

In [ ]:
display_available_tools("rf3")

### `run_rf3_prediction`

Predicts the joint 3D structure of a biomolecular complex. RF3's distinguishing capability is explicit chirality handling; outputs include per-chain pLDDT (under upstream's `chain_ptm` key), chain-pair PAE / PDE matrices for multi-chain complexes, and a composite `ranking_score`.

In [ ]:
from pathlib import Path

from proto_tools import Chain, Complex, Mmseqs2HomologySearchConfig, RF3Config, RF3Input, run_rf3_prediction
from proto_tools.entities.ligands import Fragment

In [ ]:
display_api_reference("rf3-prediction", "input", "run_rf3_prediction")

# MfnG is a tyrosine methyltransferase; L-tyrosine is its cognate substrate.
mfng_sequence = "MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDALYAYDVIDRIHDTNGFRYLLSAEARECLLPGTLFSLVGKFMHDINVAWPAWRNLAEVVRHGARDTSGAESPNGIAQEDYESLVGGINFWAPPIVTTLSRKLRASGRSGDATASVLDVGCGTGLYSQLLLREFPRWTATGLDVERIATLANAQALRLGVEERFATRAGDFWRGGWGTGYDLVLFANIFHLQTPASAVRLMRHAAACLAPDGLVAVVDQIVDADREPKTPQDRFALLFAASMTNTGGGDAYTFQEYEEWFTAAGLQRIETLDTPMHRILLARRATEPSAVPEGQASENLYFQ"
# L-tyrosine SMILES (resolves to CCD "TYR")
tyrosine_smiles = "c1cc(ccc1C[C@@H](C(=O)O)N)O"

complex = Complex(
    chains=[
        Chain(sequence=mfng_sequence, entity_type="protein"),
        Fragment(smiles=tyrosine_smiles),
    ]
)

inputs = RF3Input(complexes=[complex])

In [ ]:
display_api_reference("rf3-prediction", "config", "run_rf3_prediction")

# Reduced settings for a fast demonstration run.
config = RF3Config(
    verbose=False,
    device="cuda",  # Change to "cpu" if no GPU available (unsupported in practice)
    use_msa=True,
    msa_search_config=Mmseqs2HomologySearchConfig(search_mode="remote"),
    diffusion_batch_size=1,
)

In [ ]:
result = run_rf3_prediction(inputs, config)

In [ ]:
display_api_reference("rf3-prediction", "output", "run_rf3_prediction")

structure = result.structures[0]
m = structure.metrics

print(f"  Number of chains: {len(complex.chains)}")
print(f"  Protein length:   {len(mfng_sequence)} residues")
print(f"  ranking_score:    {m.ranking_score:.3f}")
print(f"  avg_plddt:        {m.avg_plddt:.3f}")
print(f"  ptm:              {m.ptm:.3f}")
print(f"  avg_pae:          {m.avg_pae:.3f} Å")
print(f"  pde:              {m.pde:.3f} Å")
print(f"  chain_ptm:        {m.chain_ptm}")
print(f"  has_clash:        {m.has_clash}")

#### Visualize the predicted complex

> NOTE: The 3D viewer below renders locally (JupyterLab, VS Code) but not in GitHub previews.

In [ ]:
structure.visualize(style="cartoon", color_by="bfactor")

## Export Results

Predicted structures can be exported to PDB or mmCIF format for downstream analysis in molecular visualization tools such as PyMOL, ChimeraX, or VMD. The B-factor column carries pLDDT scores for per-residue confidence visualization.

In [ ]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="rf3_cro_tyrosine", export_path=output_dir, file_format="pdb")
print("Structure exported to", sorted((output_dir / "rf3_cro_tyrosine").glob("structure_*.pdb")))